# 01 · CBIS-DDSM mass ROI preparation

Data step for the global versus patch-wise quantum-bottleneck study.

What this notebook does, top to bottom:
1. Mounts Google Drive and sets the config.
2. Lists every CBIS-DDSM series from TCIA (public, no account needed) and builds one row per mass ROI.
3. Samples whole patients up to `MAX_CASES` ROIs, so folds stay patient-level.
4. For each ROI: downloads the full mammogram + its ROI mask, makes a 256×256 square crop around the mass, saves `image.png` + `mask.png` to Drive, deletes the raw DICOMs.
5. Shows a QC grid and writes patient-level 5-fold splits.

**Resumable:** if Colab disconnects, just run all cells again; finished ROIs are skipped.
No GPU needed for this notebook (Runtime → Change runtime type → CPU is fine).


In [ ]:
# 1 · Drive + config
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT   = '/content/drive/MyDrive/quantum-placement'   # change if you want another folder
OUT_DIR   = f'{PROJECT}/data/processed/cbis_mass_256'
RAW_TMP   = '/content/raw_tmp'          # local Colab disk, cleaned after every case
MAX_CASES = 400                         # ~300-400 ROIs as in the plan
CROP_SIZE = 256
MARGIN    = 0.20                        # context around the mass bbox (20% each side)
SEED      = 42

for d in [f'{OUT_DIR}/images', f'{OUT_DIR}/masks', RAW_TMP]:
    os.makedirs(d, exist_ok=True)
print('Output ->', OUT_DIR)


In [ ]:
# 2 · Packages + helper functions
!pip -q install pydicom
import pydicom

import os, re, io, json, shutil, zipfile, random, time
import numpy as np
import pandas as pd
import cv2
import requests

TCIA_API = "https://services.cancerimagingarchive.net/nbia-api/services/v1"


# ---------- TCIA access ----------
def get_series_table(collection="CBIS-DDSM", retries=3):
    """All series in the collection, as a DataFrame (one row per series)."""
    for i in range(retries):
        try:
            r = requests.get(f"{TCIA_API}/getSeries", params={"Collection": collection}, timeout=120)
            r.raise_for_status()
            return pd.DataFrame(r.json())
        except Exception as e:
            print(f"  getSeries failed ({e}); retry {i+1}/{retries}")
            time.sleep(5 * (i + 1))
    raise RuntimeError("Could not reach TCIA. Check your internet / try again later.")


def download_series(uid, out_dir, retries=3):
    """Download one series as a zip and extract its .dcm files into out_dir/uid. Returns file list."""
    dest = os.path.join(out_dir, uid)
    if os.path.isdir(dest) and any(f.endswith(".dcm") for f in os.listdir(dest)):
        return sorted(os.path.join(dest, f) for f in os.listdir(dest) if f.endswith(".dcm"))
    os.makedirs(dest, exist_ok=True)
    for i in range(retries):
        try:
            r = requests.get(f"{TCIA_API}/getImage", params={"SeriesInstanceUID": uid}, timeout=600)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                z.extractall(dest)
            break
        except Exception as e:
            print(f"  download {uid[-12:]} failed ({e}); retry {i+1}/{retries}")
            time.sleep(5 * (i + 1))
    else:
        shutil.rmtree(dest, ignore_errors=True)
        return []
    files = []
    for root, _, fs in os.walk(dest):
        files += [os.path.join(root, f) for f in fs if f.lower().endswith(".dcm")]
    return sorted(files)


# ---------- case table ----------
ROI_PID = re.compile(r"^(Mass-(Training|Test)_P_\d+_(LEFT|RIGHT)_(CC|MLO))_(\d+)$")
FULL_PID = re.compile(r"^Mass-(Training|Test)_P_\d+_(LEFT|RIGHT)_(CC|MLO)$")


def build_case_table(series_df):
    """One row per abnormality (ROI): its full-image series UID and its ROI series UID."""
    df = series_df[series_df["PatientID"].str.startswith("Mass-")].copy()
    full = {}
    for _, r in df.iterrows():
        if FULL_PID.match(r["PatientID"]):
            full[r["PatientID"]] = r["SeriesInstanceUID"]
    rows = []
    for _, r in df.iterrows():
        m = ROI_PID.match(r["PatientID"])
        if not m:
            continue
        full_pid = m.group(1)
        if full_pid not in full:
            continue
        parts = full_pid.split("_")  # Mass-Training, P, 00001, LEFT, CC
        rows.append(dict(
            roi_id=r["PatientID"],
            full_id=full_pid,
            patient=f"P_{parts[2]}",
            split=m.group(2),
            side=m.group(3),
            view=m.group(4),
            full_uid=full[full_pid],
            roi_uid=r["SeriesInstanceUID"],
        ))
    # an ROI pid can appear in two series (cropped / mask stored separately) -> keep all uids
    t = pd.DataFrame(rows)
    t = t.groupby(["roi_id", "full_id", "patient", "split", "side", "view", "full_uid"], as_index=False)["roi_uid"].agg(list)
    return t.sort_values("roi_id").reset_index(drop=True)


def sample_cases(cases, max_cases, seed=42):
    """Pick whole patients at random until ~max_cases ROIs, so folds stay patient-level."""
    pats = sorted(cases["patient"].unique())
    random.Random(seed).shuffle(pats)
    keep, n = [], 0
    for p in pats:
        k = int((cases["patient"] == p).sum())
        if n + k > max_cases and n > 0:
            continue
        keep.append(p)
        n += k
        if n >= max_cases:
            break
    return cases[cases["patient"].isin(keep)].reset_index(drop=True)


# ---------- image processing ----------
def to_uint8(img):
    """Robust 1-99.5 percentile windowing of a full mammogram to uint8."""
    img = img.astype(np.float32)
    nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
    return (img * 255).astype(np.uint8)


def is_mask(arr):
    """CBIS ROI series hold a binary mask and a cropped image; the mask has <=2 values."""
    return np.unique(arr[:: max(1, arr.shape[0] // 256), :: max(1, arr.shape[1] // 256)]).size <= 2


def pick_mask(arrays, full_shape):
    """Among the arrays of an ROI series, return the binary mask (resized to the full image if needed)."""
    cands = [a for a in arrays if is_mask(a)]
    if not cands:
        return None
    m = max(cands, key=lambda a: a.size)
    m = (m > 0).astype(np.uint8)
    if m.shape != tuple(full_shape):  # a few CBIS masks are off by some pixels
        m = cv2.resize(m, (full_shape[1], full_shape[0]), interpolation=cv2.INTER_NEAREST)
    return m


def roi_crop(img, mask, size=256, margin=0.2):
    """Square crop around the mask bbox with a relative margin, resized to size x size."""
    ys, xs = np.where(mask > 0)
    if ys.size == 0:
        return None, None, None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    side = int(max(y1 - y0, x1 - x0) * (1 + 2 * margin))
    side = max(side, 64)
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    H, W = mask.shape
    top, left = cy - side // 2, cx - side // 2
    pad = max(0, -top, -left, top + side - H, left + side - W)
    if pad:
        img = np.pad(img, pad, mode="constant")
        mask = np.pad(mask, pad, mode="constant")
        top, left = top + pad, left + pad
    ic = img[top: top + side, left: left + side]
    mc = mask[top: top + side, left: left + side]
    ic = cv2.resize(ic, (size, size), interpolation=cv2.INTER_AREA)
    mc = cv2.resize(mc, (size, size), interpolation=cv2.INTER_NEAREST)
    return ic, (mc > 0).astype(np.uint8) * 255, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side)


In [ ]:
# 3 · Case table (cached on Drive after the first run)
cases_csv = f'{OUT_DIR}/cases_all.csv'
if os.path.exists(cases_csv):
    cases = pd.read_csv(cases_csv)
    cases['roi_uid'] = cases['roi_uid'].apply(json.loads)
else:
    series = get_series_table('CBIS-DDSM')
    print('Series in CBIS-DDSM:', len(series))
    cases = build_case_table(series)
    c2 = cases.copy(); c2['roi_uid'] = c2['roi_uid'].apply(json.dumps)
    c2.to_csv(cases_csv, index=False)

print('Mass ROIs available:', len(cases), '| patients:', cases.patient.nunique())
print(cases.split.value_counts().to_string())

subset_csv = f'{OUT_DIR}/cases_subset.csv'
if os.path.exists(subset_csv):
    subset = pd.read_csv(subset_csv)
    subset['roi_uid'] = subset['roi_uid'].apply(json.loads)
else:
    subset = sample_cases(cases, MAX_CASES, seed=SEED)
    s2 = subset.copy(); s2['roi_uid'] = s2['roi_uid'].apply(json.dumps)
    s2.to_csv(subset_csv, index=False)
print(f'Selected {len(subset)} ROIs from {subset.patient.nunique()} patients')


In [ ]:
# 4 · Download + crop (resumable)
from tqdm.auto import tqdm

def read_dicoms(files):
    out = []
    for f in files:
        try:
            out.append(pydicom.dcmread(f).pixel_array)
        except Exception as e:
            print('   unreadable', os.path.basename(f), e)
    return out

meta_csv = f'{OUT_DIR}/metadata.csv'
meta = pd.read_csv(meta_csv).to_dict('records') if os.path.exists(meta_csv) else []
done = {m['roi_id'] for m in meta}
failed = []

for full_id, grp in tqdm(subset.groupby('full_id'), desc='images'):
    todo = grp[~grp.roi_id.isin(done)]
    if todo.empty:
        continue
    full_files = download_series(grp.full_uid.iloc[0], RAW_TMP)
    full_arrs = read_dicoms(full_files)
    if not full_arrs:
        failed += list(todo.roi_id); shutil.rmtree(RAW_TMP, ignore_errors=True); os.makedirs(RAW_TMP); continue
    full = max(full_arrs, key=lambda a: a.size)
    full8 = to_uint8(full)
    for _, r in todo.iterrows():
        arrs = []
        for uid in r.roi_uid:
            arrs += read_dicoms(download_series(uid, RAW_TMP))
        mask = pick_mask(arrs, full.shape)
        ic, mc, info = roi_crop(full8, mask, CROP_SIZE, MARGIN) if mask is not None else (None, None, None)
        if ic is None:
            failed.append(r.roi_id); continue
        cv2.imwrite(f'{OUT_DIR}/images/{r.roi_id}.png', ic)
        cv2.imwrite(f'{OUT_DIR}/masks/{r.roi_id}.png', mc)
        meta.append(dict(roi_id=r.roi_id, patient=r.patient, split=r.split, side=r.side, view=r.view,
                         full_h=int(full.shape[0]), full_w=int(full.shape[1]),
                         mask_frac=round(float((mc > 0).mean()), 4), **info))
        done.add(r.roi_id)
    pd.DataFrame(meta).to_csv(meta_csv, index=False)      # save progress after every image
    shutil.rmtree(RAW_TMP, ignore_errors=True); os.makedirs(RAW_TMP)

print(f'Done: {len(done)} ROIs saved | failed: {len(failed)}')
if failed: print('Failed:', failed[:20])


In [ ]:
# 5 · QC: 12 random crops with mask outline + mask-size stats
import matplotlib.pyplot as plt
meta = pd.read_csv(meta_csv)
pick = meta.sample(min(12, len(meta)), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, (_, r) in zip(axes.ravel(), pick.iterrows()):
    im = cv2.imread(f'{OUT_DIR}/images/{r.roi_id}.png', 0)
    mk = cv2.imread(f'{OUT_DIR}/masks/{r.roi_id}.png', 0)
    rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)
    cnts, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(rgb, cnts, -1, (255, 60, 60), 1)
    ax.imshow(rgb); ax.set_title(r.roi_id.replace('Mass-', ''), fontsize=7); ax.axis('off')
plt.tight_layout(); plt.show()
print(meta.mask_frac.describe().round(3).to_string())
print('Check: masks should hug the mass; very loose or empty masks -> note them for the paper.')


In [ ]:
# 6 · Patient-level 5-fold split
from sklearn.model_selection import GroupKFold
meta = pd.read_csv(meta_csv).reset_index(drop=True)
meta['fold'] = -1
for k, (_, te) in enumerate(GroupKFold(n_splits=5).split(meta, groups=meta.patient)):
    meta.loc[te, 'fold'] = k
meta.to_csv(f'{OUT_DIR}/folds.csv', index=False)
print(meta.groupby('fold').agg(rois=('roi_id', 'size'), patients=('patient', 'nunique')).to_string())
assert meta.groupby('patient').fold.nunique().max() == 1, 'patient leaked across folds'
print('Saved folds.csv - no patient appears in two folds.')
